# DBWorld + E-mail Dataset ETL

Pipeline that converts raw MATLAB `.mat` data into cleaned Parquet/CSV.

## Data structure
- **Documents**: 64 (binary classification, label 0/1 — class semantics not specified in source)
- **4 variants**: bodies / subjects × raw / stemmed
- **Variables**: `inputs` (document-word matrix, 0/1 binary), `dictionary` (word dictionary), `words_per_doc` (total words per document), `labels` (class)

## Pipeline
1. **Extract** — load .mat files with scipy, validate schema
2. **Transform** — clean dictionary, build long (tidy) table · document metadata · wide matrices
3. **Load** — save Parquet/CSV to `processed/`
4. **QA** — reload validation, summary statistics, visualization

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

BASE_DIR = Path(r"C:\Users\USER\Desktop\[16-08-26] Cloudy\05.dbworld+e-mail")
RAW_DIR = BASE_DIR / "MATLAB"
PROCESSED_DIR = BASE_DIR / "processed"
MATRIX_DIR = PROCESSED_DIR / "matrices"
VOCAB_DIR = PROCESSED_DIR / "vocab"
for d in (PROCESSED_DIR, MATRIX_DIR, VOCAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

VARIANTS = {
    "bodies_raw": RAW_DIR / "dbworld_bodies.mat",
    "bodies_stemmed": RAW_DIR / "dbworld_bodies_stemmed.mat",
    "subjects_raw": RAW_DIR / "dbworld_subjects.mat",
    "subjects_stemmed": RAW_DIR / "dbworld_subjects_stemmed.mat",
}

## 1. Extract

Load the `.mat` files and validate the basic schema.

In [ ]:
def load_mat(path):
    """Load the document-word matrix, dictionary, and labels from a MATLAB .mat file."""
    mat = sio.loadmat(str(path))
    return {
        "inputs": np.asarray(mat["inputs"], dtype=np.int64),                  # (n_docs, n_words)
        "dictionary": np.asarray(mat["dictionary"]).ravel(),                  # (n_words,) fixed-width word list
        "words_per_doc": np.asarray(mat["words_per_doc"]).ravel().astype(int),  # total words per document
        "labels": np.asarray(mat["labels"]).ravel().astype(int),              # class labels (0/1)
    }

raw = {name: load_mat(path) for name, path in VARIANTS.items()}

for name, m in raw.items():
    print(f"{name:16s} inputs={m['inputs'].shape}  vocab={len(m['dictionary'])}  labels={m['labels'].shape}")

In [ ]:
# Schema validation: matrix columns == dictionary size, no duplicate words, labels match across files
assert all(m["inputs"].shape[1] == len(m["dictionary"]) for m in raw.values()), "matrix columns != dictionary size"
assert all(len(set(m["dictionary"])) == len(m["dictionary"]) for m in raw.values()), "duplicate words in dictionary"
ref_labels = raw["bodies_raw"]["labels"]
assert all(np.array_equal(m["labels"], ref_labels) for m in raw.values()), "label mismatch between files"

print("Schema validation passed: all 4 variants share the same 64 documents and labels")

## 2. Transform

In [ ]:
def clean_vocab(m):
    """Strip whitespace padding from the fixed-width (char) dictionary and build a word_idx → word table."""
    vocab = pd.Series([str(w).strip() for w in m["dictionary"]], name="word")
    assert vocab.str.len().gt(0).all(), "empty word found"
    assert vocab.is_unique, "duplicate words after cleaning"
    return vocab.reset_index().rename(columns={"index": "word_idx"})

vocab = {name: clean_vocab(m) for name, m in raw.items()}
vocab["bodies_raw"].head(10)

In [ ]:
def to_long(name, m, vocab_df):
    """Convert the document-word matrix to long (tidy) format (extract non-zero cells only)."""
    doc_ids, word_idxs = np.nonzero(m["inputs"])
    source, preprocess = name.split("_")
    words = vocab_df.set_index("word_idx")["word"].to_numpy()[word_idxs]
    df = pd.DataFrame({
        "doc_id": doc_ids,
        "word_idx": word_idxs,
        "word": words,
        "count": m["inputs"][doc_ids, word_idxs],
        "label": m["labels"][doc_ids],
        "words_per_doc": m["words_per_doc"][doc_ids],
    })
    df["source"] = source
    df["preprocess"] = preprocess
    return df

long = {name: to_long(name, m, vocab[name]) for name, m in raw.items()}
long_all = (pd.concat(long.values(), ignore_index=True)
            .sort_values(["source", "preprocess", "doc_id", "word_idx"])
            .reset_index(drop=True))

print(f"long table: {long_all.shape[0]:,} rows × {long_all.shape[1]} cols")
long_all.head()

In [ ]:
# Per-document metadata table (including words per doc / distinct present words per variant)
doc_meta = pd.DataFrame({"doc_id": np.arange(len(ref_labels)), "label": ref_labels})
for name, m in raw.items():
    doc_meta[f"words_per_doc_{name}"] = m["words_per_doc"]
    doc_meta[f"n_present_{name}"] = (m["inputs"] > 0).sum(axis=1)

doc_meta.head()

In [ ]:
# Wide document-word matrices (doc_id index, word columns)
matrices = {}
for name, m in raw.items():
    matrices[name] = pd.DataFrame(
        m["inputs"],
        index=pd.Index(range(len(ref_labels)), name="doc_id"),
        columns=vocab[name]["word"],
    )

matrices["bodies_raw"].iloc[:3, :8]

In [ ]:
# Dictionary comparison summary (raw vs stemmed compression, subjects inclusion)
b_raw, b_stem = set(vocab["bodies_raw"]["word"]), set(vocab["bodies_stemmed"]["word"])
s_raw, s_stem = set(vocab["subjects_raw"]["word"]), set(vocab["subjects_stemmed"]["word"])

comparison = pd.DataFrame({
    "bodies": [len(b_raw), len(b_stem), len(b_raw & b_stem)],
    "subjects": [len(s_raw), len(s_stem), len(s_raw & s_stem)],
}, index=["raw vocab size", "stemmed vocab size", "intersection (kept after stemming)"])
comparison

In [ ]:
print("subjects vocab ⊆ bodies vocab ?", s_raw.issubset(b_raw), "/", s_stem.issubset(b_stem))

## 3. Load

Save the cleaned results to `processed/` (Parquet + CSV).

In [ ]:
doc_meta.to_parquet(PROCESSED_DIR / "doc_meta.parquet")
doc_meta.to_csv(PROCESSED_DIR / "doc_meta.csv", index=False)

long_all.to_parquet(PROCESSED_DIR / "doc_term_long.parquet")

for name, df in matrices.items():
    df.to_parquet(MATRIX_DIR / f"{name}.parquet")

for name, v in vocab.items():
    v.to_csv(VOCAB_DIR / f"{name}.csv", index=False)

print("Saved:", PROCESSED_DIR)
for p in sorted(PROCESSED_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(BASE_DIR))

## 4. QA

Re-read the saved artifacts to verify integrity and inspect summary statistics.

In [ ]:
check_meta = pd.read_parquet(PROCESSED_DIR / "doc_meta.parquet")
check_long = pd.read_parquet(PROCESSED_DIR / "doc_term_long.parquet")

assert check_meta.shape == doc_meta.shape, "doc_meta mismatch"
assert check_long.shape == long_all.shape, "long table mismatch"
assert (check_meta["label"] == ref_labels).all(), "label mismatch"
print("QA reload passed")

print("\nClass distribution:")
print(check_meta["label"].value_counts().to_frame("n_docs"))

print("\nSummary by variant:")
check_long.groupby(["source", "preprocess"]).agg(
    n_rows=("count", "size"),
    n_words=("word", "nunique"),
    n_docs=("doc_id", "nunique"),
).reset_index()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

check_meta["label"].value_counts().sort_index().plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452"], rot=0)
axes[0].set_title("Class distribution (0/1)")

check_meta[["words_per_doc_bodies_raw", "words_per_doc_subjects_raw"]].plot.hist(alpha=0.6, bins=15, ax=axes[1])
axes[1].set_title("Words per doc")
axes[1].legend(["bodies", "subjects"])

check_long.groupby(["source", "preprocess"])["word"].nunique().plot.bar(ax=axes[2], rot=0)
axes[2].set_title("Vocabulary size by variant")

plt.tight_layout()
plt.show()

In [ ]:
# Top words by document frequency in bodies
top_words = (check_long[check_long["source"] == "bodies"]
             .groupby(["preprocess", "word"], as_index=False)["count"].sum()
             .sort_values(["preprocess", "count"], ascending=[True, False])
             .groupby("preprocess").head(10))

for prep, grp in top_words.groupby("preprocess"):
    print(f"--- {prep}: top 10 words by document frequency ---")
    print(grp[["word", "count"]].to_string(index=False), "\n")

## Outputs

| File | Contents |
| --- | --- |
| `processed/doc_meta.parquet` / `.csv` | Per-document metadata (64 rows: label, words per doc / distinct words per variant) |
| `processed/doc_term_long.parquet` | Combined long format of 4 variants (doc_id, word, count, label, source, preprocess) |
| `processed/matrices/{variant}.parquet` | Wide document-word matrix per variant (doc_id × word) |
| `processed/vocab/{variant}.csv` | word_idx → word dictionary per variant |

Author : Michele Filannino
